# GEM-Bench Evaluation Analysis

Qualitative and Quantitative evaluation of ad-injected response quality across different **Base LLM** and **Judge LLM** combinations.

In [34]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

OUTPUT_BASE = '/home/gpuuser7/gpuuser7_a/prateek/GEM-Bench/GemBench/benchmarking/output/evaluate_results/output'

# Experiment configurations
EXPERIMENTS = [
    {
        'base_llm': 'gemma-3-27b-it',
        'judge': 'llama-3.3-70b',
        'dir': '20260928_082937_judge_llama-3.3-70b-instruct-fp8_base_gemma-3-27b-it',
    },
    {
        'base_llm': 'gemma-3-27b-it',
        'judge': 'deepseek-r1-distill-qwen-32b',
        'dir': '20260929_175358_judge_deepseek-r1-distill-qwen-32b_base_gemma-3-27b-it',
    }
]

QUALITATIVE_METRICS = ['accuracy', 'naturalness', 'personality', 'trust', 'notice_products', 'click_products', 'identifiability']
METRIC_DISPLAY = {
    'accuracy': 'Accuracy',
    'naturalness': 'Naturalness',
    'personality': 'Personality',
    'trust': 'Trust',
    'notice_products': 'Notice',
    'click_products': 'Click',
    'identifiability': 'Identifiability',
}

QUANTITATIVE_METRICS = ['local measure', 'global_coherence', 'global measure', 'ad_content_alignment', 'has_ad', 'product_selection_accuracy', 'in_token', 'out_token']
QUANT_METRIC_DISPLAY = {
    'local measure': 'RF',
    'global_coherence': 'RC',
    'global measure': 'AF',
    'ad_content_alignment': 'AC',
    'has_ad': 'IR',
    'product_selection_accuracy': 'CTR',
    'in_token': 'In-Token',
    'out_token': 'Out-Token',
}


In [35]:
def load_results(xlsx_path):
    """Parse the evaluation_result_average.xlsx format."""
    df = pd.read_excel(xlsx_path, header=None)
    
    # Build column names from rows 1 (top-level) and 2 (metric names)
    row1 = df.iloc[1].tolist()
    row2 = df.iloc[2].tolist()
    cols = []
    for r1, r2 in zip(row1, row2):
        if pd.notna(r1) and str(r1) != 'nan':
            cols.append(str(r1))
        elif pd.notna(r2) and str(r2) != 'nan':
            cols.append(str(r2))
        else:
            cols.append('unknown')
    
    data = df.iloc[3:].reset_index(drop=True)
    data.columns = cols
    data['data_set'] = data['data_set'].ffill()
    
    # Convert metric columns to numeric
    for m in QUALITATIVE_METRICS:
        if m in data.columns:
            data[m] = pd.to_numeric(data[m], errors='coerce')
    for m in QUANTITATIVE_METRICS:
        if m in data.columns:
            data[m] = pd.to_numeric(data[m], errors='coerce')
    
    return data

# Load all experiments
all_results = []
for exp in EXPERIMENTS:
    path = f"{OUTPUT_BASE}/{exp['dir']}/evaluation_result_average.xlsx"
    df = load_results(path)
    df['base_llm'] = exp['base_llm']
    df['judge'] = exp['judge']
    all_results.append(df)

combined = pd.concat(all_results, ignore_index=True)
print(f"Loaded {len(combined)} rows from {len(EXPERIMENTS)} experiments")
print(f"Datasets: {combined['data_set'].unique().tolist()}")
print(f"Solutions: {combined['solution'].unique().tolist()}")
print(f"Base LLMs: {combined['base_llm'].unique().tolist()}")
print(f"Judges: {combined['judge'].unique().tolist()}")


Loaded 30 rows from 2 experiments
Datasets: ['CA_Prod', 'LM-Market', 'MT-Human']
Solutions: ['Ad-Chat', 'GI-R', 'GI-P', 'GIR-R', 'GIR-P']
Base LLMs: ['gemma-3-27b-it']
Judges: ['llama-3.3-70b', 'deepseek-r1-distill-qwen-32b']


In [36]:
def build_paper_table(combined_df, dataset_filter=None, solution_filter=None):
    """Build a table in the paper format: Dataset | Base LLM | Judge LLM | Solution | Metrics."""
    
    df = combined_df.copy()
    if dataset_filter:
        df = df[df['data_set'].isin(dataset_filter)]
    if solution_filter:
        df = df[df['solution'].isin(solution_filter)]
    
    # Build the table rows
    rows = []
    for ds in df['data_set'].unique():
        ds_data = df[df['data_set'] == ds]
        for base in ds_data['base_llm'].unique():
            base_data = ds_data[ds_data['base_llm'] == base]
            for judge in base_data['judge'].unique():
                judge_data = base_data[base_data['judge'] == judge]
                for sol in judge_data['solution'].unique():
                    sol_data = judge_data[judge_data['solution'] == sol]
                    row = {
                        'Dataset': ds,
                        'Base LLM': base,
                        'Judge LLM': judge,
                        'Solution': sol,
                    }
                    vals_for_overall = []
                    for m in QUALITATIVE_METRICS:
                        val = sol_data[m].values[0]
                        if pd.notna(val) and str(val).lower() != 'nan':
                            row[METRIC_DISPLAY[m]] = round(float(val), 2)
                            vals_for_overall.append(float(val))
                        else:
                            row[METRIC_DISPLAY[m]] = '—'
                    
                    if len(vals_for_overall) > 0:
                        row['Overall'] = round(sum(vals_for_overall) / len(vals_for_overall), 2)
                    else:
                        row['Overall'] = '—'
                    rows.append(row)
    
    table = pd.DataFrame(rows)
    return table

# Build full table with Ad-Chat and GIR-R (matching paper format)
def build_quant_paper_table(combined_df, dataset_filter=None, solution_filter=None):
    """Build a quantitative table in the paper format."""
    df = combined_df.copy()
    if dataset_filter:
        df = df[df['data_set'].isin(dataset_filter)]
    if solution_filter:
        df = df[df['solution'].isin(solution_filter)]
        
    rows = []
    for ds in df['data_set'].unique():
        ds_data = df[df['data_set'] == ds]
        for base in ds_data['base_llm'].unique():
            base_data = ds_data[ds_data['base_llm'] == base]
            for judge in base_data['judge'].unique():
                judge_data = base_data[base_data['judge'] == judge]
                for sol in judge_data['solution'].unique():
                    sol_data = judge_data[judge_data['solution'] == sol]
                    row = {
                        'Dataset': ds,
                        'Base LLM': base,
                        'Judge LLM': judge,
                        'Solution': sol,
                    }
                    for m in QUANTITATIVE_METRICS:
                        val = sol_data[m].values[0] if m in sol_data.columns else float('nan')
                        if pd.notna(val) and str(val).lower() != 'nan':
                            row[QUANT_METRIC_DISPLAY.get(m, m)] = round(float(val), 2)
                        else:
                            row[QUANT_METRIC_DISPLAY.get(m, m)] = '—'
                    rows.append(row)
    table = pd.DataFrame(rows)
    return table


In [37]:
def style_table(df, title=''):
    """Style a DataFrame to bold the best value per (Dataset, Solution) group for each metric."""
    metric_cols = list(METRIC_DISPLAY.values()) + ['Overall']
    
    def bold_max_by_dataset(data):
        styles = pd.DataFrame('', index=data.index, columns=data.columns)
        for ds in data['Dataset'].unique():
            ds_idx = data[data['Dataset'] == ds].index
            for col in metric_cols:
                if col in data.columns:
                    numeric_s = pd.to_numeric(data.loc[ds_idx, col], errors='coerce')
                    if numeric_s.notna().any():
                        max_val = numeric_s.max()
                        is_max = numeric_s == max_val
                        styles.loc[ds_idx[is_max], col] = 'font-weight: bold'
        return styles
    
    styled = (df.style
              .apply(bold_max_by_dataset, axis=None)
              .set_caption(title)
              .format(precision=2)
              .set_table_styles([
                  {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('padding', '10px')]},
                  {'selector': 'th', 'props': [('background-color', '#f0f0f0'), ('padding', '8px')]},
                  {'selector': 'td', 'props': [('padding', '6px 12px')]},
              ])
              .hide(axis='index'))
    return styled


In [38]:
def style_quant_table(df, title=''):
    metric_cols = list(QUANT_METRIC_DISPLAY.values()) + ['Overall']
    
    def bold_max_by_dataset(data):
        styles = pd.DataFrame('', index=data.index, columns=data.columns)
        for ds in data['Dataset'].unique():
            ds_idx = data[data['Dataset'] == ds].index
            for col in metric_cols:
                if col in data.columns:
                    numeric_s = pd.to_numeric(data.loc[ds_idx, col], errors='coerce')
                    if numeric_s.notna().any():
                        max_val = numeric_s.max()
                        is_max = numeric_s == max_val
                        styles.loc[ds_idx[is_max], col] = 'font-weight: bold'
        return styles
    
    styled = (df.style
              .apply(bold_max_by_dataset, axis=None)
              .set_caption(title)
              .format(precision=2)
              .set_table_styles([
                  {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('padding', '10px')]},
                  {'selector': 'th', 'props': [('background-color', '#f0f0f0'), ('padding', '8px')]},
                  {'selector': 'td', 'props': [('padding', '6px 12px')]},
              ])
              .hide(axis='index'))
    return styled


## Results by Base LLM and Judge LLM

Displays the Qualitative and Quantitative metric tables for each unique combination of Base LLM and Judge LLM.

In [39]:
combinations = combined[['base_llm', 'judge']].drop_duplicates().values
for base, judge in combinations:
    subset = combined[(combined['base_llm'] == base) & (combined['judge'] == judge)]
    if len(subset) > 0:
        print(f"\n{'='*60}\nBase: {base} | Judge: {judge}\n{'='*60}")
        
        # Qualitative
        qual_table = build_paper_table(subset)
        display(style_table(qual_table, title=f'Qualitative Metrics (Base: {base}, Judge: {judge})'))
        
        # Quantitative
        quant_table = build_quant_paper_table(subset)
        display(style_quant_table(quant_table, title=f'Quantitative Metrics (Base: {base}, Judge: {judge})'))



Base: gemma-3-27b-it | Judge: llama-3.3-70b


Dataset,Base LLM,Judge LLM,Solution,Accuracy,Naturalness,Personality,Trust,Notice,Click,Identifiability,Overall
CA_Prod,gemma-3-27b-it,llama-3.3-70b,Ad-Chat,61.42,39.83,51.75,31.75,66.33,84.58,34.66,52.90
CA_Prod,gemma-3-27b-it,llama-3.3-70b,GI-R,70.20,18.89,42.95,28.61,30.00,81.39,35.52,43.94
CA_Prod,gemma-3-27b-it,llama-3.3-70b,GI-P,72.11,19.79,44.05,29.09,30.58,81.65,35.82,44.73
CA_Prod,gemma-3-27b-it,llama-3.3-70b,GIR-R,76.27,32.95,54.34,33.28,46.97,81.56,33.36,51.25
CA_Prod,gemma-3-27b-it,llama-3.3-70b,GIR-P,77.48,32.81,55.95,33.97,47.77,81.82,33.27,51.87
LM-Market,gemma-3-27b-it,llama-3.3-70b,Ad-Chat,82.85,56.08,70.43,53.01,66.61,81.08,37.87,63.99
LM-Market,gemma-3-27b-it,llama-3.3-70b,GI-R,88.98,64.62,80.38,73.55,52.42,75.00,52.30,69.61
LM-Market,gemma-3-27b-it,llama-3.3-70b,GI-P,89.78,63.06,80.27,75.27,54.52,74.73,49.05,69.53
LM-Market,gemma-3-27b-it,llama-3.3-70b,GIR-R,89.46,67.26,77.74,71.45,60.65,76.45,47.32,70.05
LM-Market,gemma-3-27b-it,llama-3.3-70b,GIR-P,89.68,66.56,76.45,70.81,60.00,75.65,43.13,68.90


Dataset,Base LLM,Judge LLM,Solution,RF,RC,AF,AC,IR,CTR,In-Token,Out-Token
CA_Prod,gemma-3-27b-it,llama-3.3-70b,Ad-Chat,27.56,51.94,79.85,36.45,100.00,41.67,2973.21,493.15
CA_Prod,gemma-3-27b-it,llama-3.3-70b,GI-R,37.60,67.24,83.62,60.38,100.00,29.78,5.09,174.44
CA_Prod,gemma-3-27b-it,llama-3.3-70b,GI-P,37.61,68.16,84.33,59.83,100.00,35.26,5.12,174.38
CA_Prod,gemma-3-27b-it,llama-3.3-70b,GIR-R,43.01,70.86,83.33,56.95,100.00,29.78,393.45,421.17
CA_Prod,gemma-3-27b-it,llama-3.3-70b,GIR-P,42.94,70.95,83.18,56.73,100.00,35.26,378.95,418.76
LM-Market,gemma-3-27b-it,llama-3.3-70b,Ad-Chat,30.39,52.45,84.23,46.24,98.57,—,929.55,831.05
LM-Market,gemma-3-27b-it,llama-3.3-70b,GI-R,35.23,57.92,79.99,48.48,100.00,—,23.23,675.45
LM-Market,gemma-3-27b-it,llama-3.3-70b,GI-P,35.17,57.78,79.76,46.58,100.00,—,23.23,675.45
LM-Market,gemma-3-27b-it,llama-3.3-70b,GIR-R,32.70,55.00,76.91,47.46,100.00,—,759.32,1384.13
LM-Market,gemma-3-27b-it,llama-3.3-70b,GIR-P,32.57,54.88,75.77,45.41,100.00,—,758.16,1382.38



Base: gemma-3-27b-it | Judge: deepseek-r1-distill-qwen-32b


Dataset,Base LLM,Judge LLM,Solution,Accuracy,Naturalness,Personality,Trust,Notice,Click,Identifiability,Overall
CA_Prod,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,Ad-Chat,58.00,61.83,50.67,31.83,39.67,73.75,33.37,49.87
CA_Prod,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GI-R,61.89,43.28,41.64,32.70,24.26,66.56,32.82,43.31
CA_Prod,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GI-P,63.22,43.60,42.48,32.15,23.80,66.03,33.36,43.52
CA_Prod,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GIR-R,72.34,60.04,52.38,33.93,25.70,68.65,33.18,49.46
CA_Prod,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GIR-P,73.26,60.83,52.73,34.71,25.54,68.68,33.09,49.83
LM-Market,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,Ad-Chat,84.73,63.01,61.51,46.08,56.02,71.40,35.38,59.73
LM-Market,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GI-R,86.61,59.41,76.13,56.77,42.10,60.54,37.35,59.84
LM-Market,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GI-P,87.04,59.84,75.70,54.73,40.75,61.88,33.63,59.08
LM-Market,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GIR-R,87.53,65.38,70.86,55.48,51.08,63.12,35.34,61.25
LM-Market,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GIR-P,87.74,64.52,69.68,55.11,51.13,63.49,35.04,60.96


Dataset,Base LLM,Judge LLM,Solution,RF,RC,AF,AC,IR,CTR,In-Token,Out-Token
CA_Prod,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,Ad-Chat,27.56,51.94,79.85,36.45,100.00,41.67,2973.21,493.15
CA_Prod,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GI-R,37.60,67.24,83.62,60.38,100.00,29.78,5.09,174.44
CA_Prod,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GI-P,37.61,68.16,84.33,59.83,100.00,35.26,5.12,174.38
CA_Prod,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GIR-R,43.01,70.86,83.33,56.95,100.00,29.78,393.45,421.17
CA_Prod,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GIR-P,42.94,70.95,83.18,56.73,100.00,35.26,378.95,418.76
LM-Market,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,Ad-Chat,30.39,52.45,84.23,46.24,98.57,—,929.55,831.05
LM-Market,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GI-R,35.23,57.92,79.99,48.48,100.00,—,23.23,675.45
LM-Market,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GI-P,35.17,57.78,79.76,46.58,100.00,—,23.23,675.45
LM-Market,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GIR-R,32.70,55.00,76.91,47.46,100.00,—,759.32,1384.13
LM-Market,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GIR-P,32.57,54.88,75.77,45.41,100.00,—,758.16,1382.38


In [40]:
# Winner by Overall Avg per Dataset per Judge
# Qualitative: excludes identifiability | Quantitative: excludes in_token, out_token

QUAL_NO_IDENT = ['accuracy', 'naturalness', 'personality', 'trust', 'notice_products', 'click_products']
QUANT_NO_TOKEN = ['local measure', 'global_coherence', 'global measure', 'ad_content_alignment', 'has_ad', 'product_selection_accuracy']
DATASET_ORDER = ['MT-Human', 'LM-Market', 'CA_Prod']
SOLUTION_ORDER = ['Ad-Chat', 'GI-R', 'GI-P', 'GIR-R', 'GIR-P']

def get_winners(df, metrics):
    rows = []
    for ds in DATASET_ORDER:
        best_sol, best_score = None, -1
        for sol in SOLUTION_ORDER:
            sub = df[(df['data_set'] == ds) & (df['solution'] == sol)]
            if sub.empty:
                continue
            vals = [float(sub[m].values[0]) for m in metrics if m in sub.columns and pd.notna(sub[m].values[0])]
            avg = sum(vals) / len(vals) if vals else 0
            if avg > best_score:
                best_score, best_sol = avg, sol
        rows.append({'Dataset': ds, 'Winner': best_sol, 'Overall Avg': round(best_score, 2)})
    return pd.DataFrame(rows)


In [41]:
# Judge: llama-3.3-70b
judge_df = combined[combined['judge'] == 'llama-3.3-70b']

print("Judge: llama-3.3-70b")
print("=" * 40)
print("\nQualitative Winner (excl. Identifiability)")
display(get_winners(judge_df, QUAL_NO_IDENT))
print("\nQuantitative Winner (excl. Token Counts)")
display(get_winners(judge_df, QUANT_NO_TOKEN))


Judge: llama-3.3-70b

Qualitative Winner (excl. Identifiability)


,Dataset,Winner,Overall Avg
0,MT-Human,GI-R,79.17
1,LM-Market,GIR-R,73.84
2,CA_Prod,Ad-Chat,55.94



Quantitative Winner (excl. Token Counts)


,Dataset,Winner,Overall Avg
0,MT-Human,GI-R,63.74
1,LM-Market,GI-R,64.32
2,CA_Prod,GIR-P,64.85


In [42]:
# Judge: deepseek-r1
judge_df = combined[combined['judge'] == 'deepseek-r1-distill-qwen-32b']

print("Judge: deepseek-r1")
print("=" * 40)
print("\nQualitative Winner (excl. Identifiability)")
display(get_winners(judge_df, QUAL_NO_IDENT))
print("\nQuantitative Winner (excl. Token Counts)")
display(get_winners(judge_df, QUANT_NO_TOKEN))


Judge: deepseek-r1

Qualitative Winner (excl. Identifiability)


,Dataset,Winner,Overall Avg
0,MT-Human,GI-R,69.50
1,LM-Market,GIR-R,65.57
2,CA_Prod,Ad-Chat,52.62



Quantitative Winner (excl. Token Counts)


,Dataset,Winner,Overall Avg
0,MT-Human,GI-R,63.74
1,LM-Market,GI-R,64.32
2,CA_Prod,GIR-P,64.85


In [43]:
# Identifiability Analysis per Judge per Dataset
# Lower = ad is more covert | Higher = ad is more recognizable

print("Identifiability Scores by Solution")
print("=" * 50)

for judge_name, judge_key in [('llama-3.3-70b', 'llama-3.3-70b'), ('deepseek-r1', 'deepseek-r1-distill-qwen-32b')]:
    judge_df = combined[combined['judge'] == judge_key]
    rows = []
    for ds in DATASET_ORDER:
        for sol in SOLUTION_ORDER:
            sub = judge_df[(judge_df['data_set'] == ds) & (judge_df['solution'] == sol)]
            if sub.empty:
                continue
            val = sub['identifiability'].values[0] if 'identifiability' in sub.columns and pd.notna(sub['identifiability'].values[0]) else None
            rows.append({'Dataset': ds, 'Solution': sol, 'Identifiability': round(float(val), 2) if val else None})
    df = pd.DataFrame(rows).pivot(index='Solution', columns='Dataset', values='Identifiability')[DATASET_ORDER]
    df['Avg'] = df.mean(axis=1).round(2)
    print(f"\nJudge: {judge_name}")
    display(df)


Identifiability Scores by Solution

Judge: llama-3.3-70b


Dataset,MT-Human,LM-Market,CA_Prod,Avg
Solution,,,,
Ad-Chat,48.60,37.87,34.66,40.38
GI-P,61.90,49.05,35.82,48.92
GI-R,58.63,52.30,35.52,48.82
GIR-P,49.67,43.13,33.27,42.02
GIR-R,49.70,47.32,33.36,43.46



Judge: deepseek-r1


Dataset,MT-Human,LM-Market,CA_Prod,Avg
Solution,,,,
Ad-Chat,46.37,35.38,33.37,38.37
GI-P,44.20,33.63,33.36,37.06
GI-R,44.20,37.35,32.82,38.12
GIR-P,34.17,35.04,33.09,34.10
GIR-R,36.40,35.34,33.18,34.97


In [44]:
# GI-R vs GI-P Comparison

DATASET_ORDER = ['MT-Human', 'LM-Market', 'CA_Prod']

for judge in combined['judge'].unique():
    judge_df = combined[combined['judge'] == judge]
    print(f"Judge: {judge}")
    print("=" * 50)
    
    for ds in DATASET_ORDER:
        sub = judge_df[(judge_df['data_set'] == ds) & (judge_df['solution'].isin(['GI-R', 'GI-P']))]
        if sub.empty:
            continue
        
        # Qualitative
        qual_cols = ['accuracy', 'naturalness', 'personality', 'trust', 'notice_products', 'click_products', 'identifiability']
        qual_display = {'accuracy': 'Acc', 'naturalness': 'Nat', 'personality': 'Per', 'trust': 'Tru',
                        'notice_products': 'Not', 'click_products': 'Cli', 'identifiability': 'Idt'}
        qual = sub[['solution'] + qual_cols].set_index('solution').rename(columns=qual_display).reindex(['GI-R', 'GI-P']).round(2)
        print(f"\n  {ds} - Qualitative")
        display(qual)
        gi_r_avg = qual.loc['GI-R'].mean()
        gi_p_avg = qual.loc['GI-P'].mean()
        winner = 'GI-R' if gi_r_avg >= gi_p_avg else 'GI-P'
        print(f"  → Winner: {winner} (GI-R avg={gi_r_avg:.2f}, GI-P avg={gi_p_avg:.2f})")
        
        # Quantitative
        quant_cols = [c for c in ['local measure', 'global_coherence', 'global measure', 'ad_content_alignment', 'has_ad', 'product_selection_accuracy'] if c in sub.columns]
        quant_display = {'local measure': 'RF', 'global_coherence': 'RC', 'global measure': 'AF',
                         'ad_content_alignment': 'AC', 'has_ad': 'IR', 'product_selection_accuracy': 'CTR'}
        quant = sub[['solution'] + quant_cols].set_index('solution').rename(columns=quant_display).reindex(['GI-R', 'GI-P']).round(2)
        print(f"  {ds} - Quantitative")
        display(quant)
        gi_r_avg_q = quant.loc['GI-R'].mean()
        gi_p_avg_q = quant.loc['GI-P'].mean()
        winner_q = 'GI-R' if gi_r_avg_q >= gi_p_avg_q else 'GI-P'
        print(f"  → Winner: {winner_q} (GI-R avg={gi_r_avg_q:.2f}, GI-P avg={gi_p_avg_q:.2f})")
    print()


Judge: llama-3.3-70b

  MT-Human - Qualitative


,Acc,Nat,Per,Tru,Not,Cli,Idt
solution,,,,,,,
GI-R,90.0,76.0,88.0,86.0,60.0,75.0,58.63
GI-P,90.0,70.0,86.0,85.0,58.0,71.0,61.90


  → Winner: GI-R (GI-R avg=76.23, GI-P avg=74.56)
  MT-Human - Quantitative


,RF,RC,AF,AC,IR,CTR
solution,,,,,,
GI-R,37.24,59.20,80.13,42.12,100.0,NaN
GI-P,36.97,59.04,79.59,38.26,100.0,NaN


  → Winner: GI-R (GI-R avg=63.74, GI-P avg=62.77)

  LM-Market - Qualitative


...

  → Winner: GI-R (GI-R avg=64.32, GI-P avg=63.86)

  CA_Prod - Qualitative


,Acc,Nat,Per,Tru,Not,Cli,Idt
solution,,,,,,,
GI-R,61.89,43.28,41.64,32.70,24.26,66.56,32.82
GI-P,63.22,43.60,42.48,32.15,23.80,66.03,33.36


  → Winner: GI-P (GI-R avg=43.31, GI-P avg=43.52)
  CA_Prod - Quantitative


,RF,RC,AF,AC,IR,CTR
solution,,,,,,
GI-R,37.60,67.24,83.62,60.38,100.0,29.78
GI-P,37.61,68.16,84.33,59.83,100.0,35.26


  → Winner: GI-P (GI-R avg=63.10, GI-P avg=64.20)

